<div style="background-color:#0e2233;background-image:linear-gradient(135deg,#0b1a29,#123049);color:#ffffff;padding:26px 28px 24px;border-radius:12px;border:1px solid #2a4a66;">
<div style="color:#a0d2ff;font-size:0.78em;font-weight:600;letter-spacing:0.18em;">AI &amp; ROBOTICS · NHÁNH AI ENGINEER · NOTEBOOK THỰC HÀNH</div>
<div style="width:56px;height:4px;background-color:#E07A1F;margin:14px 0 12px;"></div>
<h1 style="color:#ffffff;font-size:2.3em;line-height:1.15;margin:0;border:0;">Chương 17 · Cây quyết định</h1>
<p style="color:#cfe6ff;font-size:1.12em;line-height:1.5;margin:10px 0 0;">Một chuỗi câu hỏi có hoặc không: Gini tính tay, độ sâu và overfitting, cây hồi quy, độ quan trọng đặc trưng, và vì sao một cây không ổn định.</p>
</div>

### Sau notebook này bạn sẽ

- Đọc cây và tính tay Gini để chọn câu hỏi.
- Hãm cây bằng độ sâu tối đa, chọn bằng cross-validation.
- Dùng cây cho hồi quy và đọc độ quan trọng đặc trưng.
- Thấy cây đổi cấu trúc khi đổi một ít dữ liệu.

<div>
<span style="display:inline-block;margin:4px 6px 0 0;padding:4px 12px;border-radius:99px;background-color:#FDF1E6;color:#2b2b2b;border:1px solid #E07A1F;font-size:0.9em;"><b>Thời gian chạy:</b> dưới 1 phút</span>
<span style="display:inline-block;margin:4px 6px 0 0;padding:4px 12px;border-radius:99px;background-color:#FDF1E6;color:#2b2b2b;border:1px solid #E07A1F;font-size:0.9em;"><b>Runtime:</b> CPU là đủ, không cần GPU</span>
<span style="display:inline-block;margin:4px 6px 0 0;padding:4px 12px;border-radius:99px;background-color:#FDF1E6;color:#2b2b2b;border:1px solid #E07A1F;font-size:0.9em;"><b>Trang web của chương:</b> <a href="https://ai-and-robotics-handbook.vercel.app/ai-engineer/ch17-cay-quyet-dinh/01-chuoi-cau-hoi/" style="color:#1F5F8B;">mở bài đọc</a></span>
</div>

### Mục lục

1. [Trang 1 · Một chuỗi câu hỏi](#trang-1)
2. [Trang 2 · Gini và cách chọn câu hỏi](#trang-2)
3. [Trang 3 · Độ sâu và overfitting](#trang-3)
4. [Trang 4 · Cây hồi quy, độ quan trọng và tính bất ổn](#trang-4)
5. [Trang 5 · Bài tập](#trang-5)

### Cách chạy

Bấm vào một ô code rồi nhấn **Shift + Enter** để chạy ô đó và nhảy sang ô kế tiếp. Muốn chạy hết một lượt thì chọn menu **Runtime > Run all**. Các ô nên chạy theo thứ tự từ trên xuống, vì ô sau dùng biến của ô trước.

<a name="trang-1" id="trang-1"></a>

## Trang 1 · Một chuỗi câu hỏi

Cây của 10 bạn, và cây không cần chuẩn hóa.

*Đọc kèm trên web: [Trang 1 · Một chuỗi câu hỏi](https://ai-and-robotics-handbook.vercel.app/ai-engineer/ch17-cay-quyet-dinh/01-chuoi-cau-hoi/)*

In [ ]:
import numpy as np                                                  # tính toán
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor, export_text   # cây
X = np.array([[1, 7], [2, 6], [3, 8], [4, 7], [5, 6], [2, 7], [6, 8], [3, 5], [7, 7], [4, 6]], dtype=float)   # (giờ học, giờ ngủ)
y = (np.array([2, 4, 5, 7, 6, 3.5, 8.5, 4, 9, 6.5]) >= 5).astype(int)   # 1 là đạt
cay = DecisionTreeClassifier(random_state=0).fit(X, y)              # để cây mọc tự do
print(export_text(cay, feature_names=["gio", "ngu"]))               # in cây dạng chữ
print(cay.get_depth(), cay.get_n_leaves())                          # độ sâu, số lá

Output mong đợi:

```text
|--- gio <= 3.50
|   |--- ngu <= 7.50
|   |   |--- class: 0
|   |--- ngu >  7.50
|   |   |--- class: 1
|--- gio >  3.50
|   |--- class: 1

2 3
```

In [ ]:
from sklearn.datasets import load_breast_cancer                    # ung thư vú
from sklearn.model_selection import train_test_split, cross_val_score   # chia và cross-validation
from sklearn.preprocessing import StandardScaler                   # chuẩn hóa
dd = load_breast_cancer()                                          # 569 ca
Xb, yb = dd.data, 1 - dd.target                                    # 1 là ác tính
Xtr, Xte, ytr, yte = train_test_split(Xb, yb, test_size=0.25, random_state=0, stratify=yb)   # cách chia Chương 12
s = StandardScaler().fit(Xtr)                                      # bộ chuẩn hóa
m_goc = DecisionTreeClassifier(max_depth=3, random_state=0).fit(Xtr, ytr)              # dữ liệu gốc
m_chuan = DecisionTreeClassifier(max_depth=3, random_state=0).fit(s.transform(Xtr), ytr)   # dữ liệu đã chuẩn hóa
print((m_goc.predict(Xte) == m_chuan.predict(s.transform(Xte))).all())                 # dự đoán có trùng nhau không

Output mong đợi:

```text
True
```

<div style="border-left:6px solid #E07A1F;background-color:#FDF1E6;color:#2b2b2b;padding:12px 18px 14px;border-radius:8px;margin:10px 0;">
<div style="color:#9A4F0E;font-weight:700;font-size:0.85em;letter-spacing:0.08em;">VÌ SAO LẠI TRÙNG NHAU 100%?</div>
<p style="margin:8px 0 0;color:#2b2b2b;line-height:1.6;">Câu hỏi "giá trị có nhỏ hơn ngưỡng không" chỉ phụ thuộc thứ tự các giá trị. Chuẩn hóa giữ nguyên thứ tự, nên cây y hệt.</p>
</div>

<a name="trang-2" id="trang-2"></a>

## Trang 2 · Gini và cách chọn câu hỏi

Thử mọi câu hỏi, chọn câu làm Gini giảm nhiều nhất.

*Đọc kèm trên web: [Trang 2 · Gini và cách chọn câu hỏi](https://ai-and-robotics-handbook.vercel.app/ai-engineer/ch17-cay-quyet-dinh/02-gini-va-chon-cau-hoi/)*

In [ ]:
gini = lambda v: 1 - sum(np.mean(v == k) ** 2 for k in (0, 1)) if len(v) else 0.0   # công thức (17.1)
ung_vien = []                                                  # mọi phép chia thử được
for j, ten in [(0, "gio"), (1, "ngu")]:                        # hai đặc trưng
    gt = np.unique(X[:, j])                                    # giá trị khác nhau, đã sắp xếp
    for t in (gt[:-1] + gt[1:]) / 2:                           # ngưỡng là trung điểm
        L, R = y[X[:, j] <= t], y[X[:, j] > t]                 # hai nhóm con
        g = (len(L) * gini(L) + len(R) * gini(R)) / len(y)     # Gini trung bình có trọng số (17.2)
        ung_vien.append((round(float(g), 4), ten, float(t), len(L), len(R)))   # lưu lại
print(round(gini(y), 4))                                       # Gini của nút gốc
for u in sorted(ung_vien)[:4]:                                 # bốn phép chia tốt nhất
    print(u)                                                   # (Gini, đặc trưng, ngưỡng, trái, phải)

Output mong đợi:

```text
0.48
(0.16, 'gio', 3.5, 5, 5)
(0.1714, 'gio', 2.5, 3, 7)
(0.3429, 'gio', 4.5, 7, 3)
(0.4, 'gio', 1.5, 1, 9)
```

<a name="trang-3" id="trang-3"></a>

## Trang 3 · Độ sâu và overfitting

Độ sâu tối đa và cross-validation trên ung thư vú.

*Đọc kèm trên web: [Trang 3 · Độ sâu và overfitting](https://ai-and-robotics-handbook.vercel.app/ai-engineer/ch17-cay-quyet-dinh/03-do-sau-va-overfitting/)*

In [ ]:
for md in [1, 2, 3, 4, 5, None]:                                   # độ sâu tăng dần
    m = DecisionTreeClassifier(max_depth=md, random_state=0).fit(Xtr, ytr)   # huấn luyện
    cv = cross_val_score(DecisionTreeClassifier(max_depth=md, random_state=0), Xtr, ytr, cv=5).mean()   # cross-validation
    print(md, m.get_n_leaves(), round(m.score(Xtr, ytr), 4), round(cv, 4), round(m.score(Xte, yte), 4))   # lá, train, CV, test

Output mong đợi:

```text
1 2 0.9249 0.9014 0.8741
2 4 0.9507 0.9201 0.9231
3 8 0.9742 0.9295 0.958
4 13 0.993 0.9201 0.9441
5 16 0.9953 0.9201 0.958
None 18 1.0 0.9178 0.9441
```

<div style="border-left:6px solid #2E8B57;background-color:#EAF6EF;color:#2b2b2b;padding:12px 18px 14px;border-radius:8px;margin:10px 0;">
<div style="color:#1E6B42;font-weight:700;font-size:0.85em;letter-spacing:0.08em;">ĐỌC KẾT QUẢ</div>
<p style="margin:8px 0 0;color:#2b2b2b;line-height:1.6;">Cross-validation chọn độ sâu 3 (0,930). Từ độ sâu 4, train vẫn tăng tới 100% mà cross-validation giảm: overfitting.</p>
</div>

<a name="trang-4" id="trang-4"></a>

## Trang 4 · Cây hồi quy, độ quan trọng và tính bất ổn

Cây hồi quy, độ quan trọng, tính bất ổn.

*Đọc kèm trên web: [Trang 4 · Cây hồi quy, độ quan trọng và tính bất ổn](https://ai-and-robotics-handbook.vercel.app/ai-engineer/ch17-cay-quyet-dinh/04-hoi-quy-va-do-quan-trong/)*

In [ ]:
from sklearn.datasets import load_diabetes                          # bộ của Chương 9
from sklearn.metrics import r2_score                                # R²
Xd, yd = load_diabetes(return_X_y=True, as_frame=True, scaled=False)   # đơn vị gốc
Xdtr, Xdte, ydtr, ydte = train_test_split(Xd, yd, test_size=0.2, random_state=42)   # cách chia Chương 9
for md in [2, 3, 5, None]:                                          # bốn độ sâu
    r = DecisionTreeRegressor(max_depth=md, random_state=0).fit(Xdtr, ydtr)   # cây hồi quy
    print(md, round(r2_score(ydte, r.predict(Xdte)), 4))            # R² test
m3 = DecisionTreeClassifier(max_depth=3, random_state=0).fit(Xtr, ytr)   # cây sâu 3 trên ung thư vú
qt = m3.feature_importances_                                       # độ quan trọng
o = np.argsort(qt)[::-1][:4]                                       # bốn đặc trưng quan trọng nhất
print([(str(dd.feature_names[i]), round(float(qt[i]), 4)) for i in o], int((qt > 0).sum()))   # tên, độ quan trọng, số đặc trưng được dùng

Output mong đợi:

```text
2 0.2949
3 0.3294
5 0.2602
None -0.0063
[('mean concave points', 0.7731), ('worst concavity', 0.0659), ('worst radius', 0.0653), ('mean texture', 0.037)] 7
```

In [ ]:
rng = np.random.default_rng(0)                                         # seed
for k in range(5):                                                     # năm lần thử
    idx = rng.choice(len(Xtr), len(Xtr) - 20, replace=False)           # bỏ ngẫu nhiên 20 ca
    t = DecisionTreeClassifier(random_state=0).fit(Xtr[idx], ytr[idx]).tree_   # cây không giới hạn
    goc, trai, phai = t.feature[0], t.feature[1], t.feature[t.children_right[0]]   # đặc trưng ở gốc và hai nút con
    print(k, dd.feature_names[goc], "|", dd.feature_names[trai], "|", dd.feature_names[phai])   # in tên

Output mong đợi:

```text
0 mean concave points | worst radius | worst perimeter
1 mean concave points | worst area | worst concavity
2 mean concave points | worst radius | worst concavity
3 mean concave points | worst radius | worst concavity
4 mean concave points | worst radius | worst concavity
```

<div style="border-left:6px solid #7A6A58;background-color:#F4F1EC;color:#2b2b2b;padding:12px 18px 14px;border-radius:8px;margin:10px 0;">
<div style="color:#5C4E3F;font-weight:700;font-size:0.85em;letter-spacing:0.08em;">THỬ SỬA</div>
<p style="margin:8px 0 0;color:#2b2b2b;line-height:1.6;"><b>1.</b> Bỏ 100 ca thay vì 20. Câu hỏi ở gốc còn giữ nguyên không?</p>
<p style="margin:8px 0 0;color:#2b2b2b;line-height:1.6;"><b>2.</b> Thêm min_samples_leaf=10 vào cây. Tầng hai có ổn định hơn không?</p>
</div>

<a name="trang-5" id="trang-5"></a>

## Trang 5 · Bài tập

Ba bài tập. Tính tay trước, rồi mới mở đáp án.

*Đọc kèm trên web: [Trang 5 · Bài tập](https://ai-and-robotics-handbook.vercel.app/ai-engineer/ch17-cay-quyet-dinh/05-bai-tap/)*

### Bài 17.1 · Câu hỏi về giờ ngủ

Tính tay Gini của phép chia "giờ ngủ ≤ 5,5".
<details><summary>Xem đáp án</summary>

Trái 1 bạn ($G = 0$), phải 9 bạn ($G \approx 0{,}444$). Trung bình có trọng số 0,4: chỉ cắt riêng một điểm, nhóm còn lại vẫn lẫn lộn.
</details>

### Bài 17.2 · Đi theo cây

Đi theo cây để đoán bạn mới (3,4 giờ học; 6,3 giờ ngủ), so với KNN $k = 3$.
<details><summary>Xem đáp án</summary>

Giờ học ≤ 3,5 rồi giờ ngủ ≤ 7,5: không đạt, "chắc chắn" 100%. KNN $k = 3$ đoán đạt. Code ở ô ngay dưới.
</details>

In [ ]:
print(cay.predict([[3.4, 6.3]]), cay.predict_proba([[3.4, 6.3]]))   # nhãn và xác suất của cây

Output mong đợi:

```text
[0] [[1. 0.]]
```

### Bài 17.3 · Entropy

Tính entropy của nút gốc, hai nhóm con của "giờ học ≤ 3,5", và information gain.
<details><summary>Xem đáp án</summary>

0,971; 0,722 và 0; information gain $\approx 0{,}61$ bit. Code ở ô ngay dưới.
</details>

In [ ]:
H = lambda p: -sum(q * np.log2(q) for q in p if q > 0)                         # entropy theo bit
print(round(H([0.6, 0.4]), 4), round(H([0.2, 0.8]), 4), round(H([0.6, 0.4]) - 0.5 * H([0.2, 0.8]), 4))   # gốc, nhóm trái, information gain

Output mong đợi:

```text
0.971 0.7219 0.61
```